# 03 - RFM Segmentation
**Question:** who are the most valuable customers, and who has gone quiet?

Source: `marts.mart_customer_rfm`. Recency is measured at the day after the window ends; frequency is bucketed 1 / 2 / 3+ because most customers order once.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from ingestion.config import get_warehouse_path  # noqa: E402

con = duckdb.connect(str(get_warehouse_path()), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the warehouse and return a DataFrame."""
    return con.execute(sql).df()

## Segment summary

In [ ]:
rfm = q("SELECT * FROM marts.mart_customer_rfm")
seg = (
    rfm.groupby("segment")
    .agg(customers=("customer_key", "count"), revenue=("monetary", "sum"),
         avg_spend=("monetary", "mean"), avg_recency_days=("recency_days", "mean"))
    .reset_index()
)
seg["customer_share_pct"] = 100 * seg["customers"] / seg["customers"].sum()
seg["revenue_share_pct"] = 100 * seg["revenue"] / seg["revenue"].sum()
seg = seg.sort_values("revenue", ascending=False).reset_index(drop=True)
display(seg)

In [ ]:
long = seg.melt(id_vars="segment", value_vars=["customer_share_pct", "revenue_share_pct"],
                var_name="measure", value_name="percent")
px.bar(long, x="segment", y="percent", color="measure", barmode="group",
       title="Share of customers vs share of revenue by segment").show()

## Concentration: how much revenue comes from the top customers?

In [ ]:
ranked = rfm.sort_values("monetary", ascending=False).reset_index(drop=True)
total = ranked["monetary"].sum()
conc = pd.DataFrame({
    "top_customers": ["top 1%", "top 5%", "top 10%", "top 20%"],
    "revenue_share_pct": [100 * ranked["monetary"].head(max(1, int(len(ranked) * p))).sum() / total
                          for p in (0.01, 0.05, 0.10, 0.20)],
})
display(conc)

## Frequency and recency distribution

In [ ]:
display(rfm["frequency"].value_counts().sort_index().rename("customers").to_frame())
px.histogram(rfm, x="recency_days", nbins=40, title="Days since last order (at window end)").show()

## How to read this
- Segment rules live in `mart_customer_rfm.sql`. If a segment is very small or very large, revisit the rules rather than force the story.
- Because almost all customers have frequency 1, most of the differentiation comes from recency and spend. Say so when presenting.
- Suggested actions: win back the 'Lapsed high-value' group, nurture 'Recent high-value' customers toward a second order.